# Joint survival models with original lifetime Oλ
Fit the adjusted continuous and midpoint-transition models for GEPs + Oλ and atlas cell types + Oλ. Reuse the feature-only midpoint crossovers. Notebook 02 supplies the lifetime Oλ crossover.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import tempfile
import numpy as np
import pandas as pd

drive.mount('/content/drive')
DATA = Path('/content/drive/MyDrive/Hartert/CAR-T-spectral-organization/DATA')
RUN = ['gep', 'atlas']
# Set only if the previous O-lambda notebook produced several compatible runs.
OLAMBDA_TRANSITION_CSV = None
TCAT = DATA/'patient_TCAT_particles'
ATLAS = DATA/'Functional_CART_atlas_patient_particles_hard_labels_all8'
O = DATA/'olambda_asymptotes_min1000_no_floor_v1'
oe, op = O/'patient_olambda_asymptotes.csv', O/'patient_olambda_particles.csv'
outcomes = DATA/'clinical_outcomes.csv'
inputs = {
    'gep': (TCAT/'all_patient_TCAT_estimates_harmonized.csv', TCAT/'all_patient_TCAT_particles_harmonized.npz'),
    'atlas': (ATLAS/'patient_atlas_estimates.csv', ATLAS/'patient_atlas_particles.npz'),
}
O_ROOT = DATA/'bayesian_Olambda_survival'/'lifetime_no_floor_adjusted_midpoint_v1'/'smooth_transition_Q10_Q90_midpoint'
OUTPUT = DATA/'lifetime_no_floor_TCAT_atlas_joint_survival_v1'
draws, tune, chains, cores = 2000, 2000, 4, 4
width, target_accept, seed = 0.10, 0.99, 0
adjustments = {'Treatment':'axicel','Disease':'lymphoma'}
assert RUN and set(RUN) <= inputs.keys() and len(set(RUN)) == len(RUN)
FEATURE_CROSSOVERS = {}  # Optional explicit CSV paths keyed by 'gep' and 'atlas'.
FEATURE_ROOT = DATA / 'direct_reciprocal_TCAT_atlas_survival_v1'

In [ ]:
revision = '42949bfb686819b08054293d42bea6e671452a29'
repo = Path(tempfile.mkdtemp(prefix='joint-midpoint-'))/'repo'
subprocess.run(['git','clone','-q','--single-branch','--branch','olambda-lifetime-no-floor',
                'https://github.com/adamnik/CAR-T-spectral-organization.git',repo],check=True)
subprocess.run(['git','-C',repo,'checkout','-q','--detach',revision],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pymc==5.28.5','arviz>=0.22,<1','h5netcdf>=1.5'],check=True)
print('Joint model revision:', revision)

In [ ]:
def digest(path):
    with open(path,'rb') as stream:
        h = hashlib.sha256()
        for block in iter(lambda: stream.read(1024*1024), b''):
            h.update(block)
        return h.hexdigest()


def run_live(command):
    print(' '.join(map(str,command)), flush=True)
    with subprocess.Popen(command,cwd=repo,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          text=True,bufsize=1) as proc:
        for line in proc.stdout:
            print(line,end='',flush=True)
        if proc.wait():
            raise subprocess.CalledProcessError(proc.returncode,command)

from functools import lru_cache
file_hash = lru_cache(maxsize=None)(digest)
common = ['--minimum-cells','1000','--draws',str(draws),'--tune',str(tune),'--chains',str(chains),
          '--cores',str(cores),'--target-accept',str(target_accept),'--random-state',str(seed),
          '--adjustment','Treatment=axicel','--adjustment','Disease=lymphoma']
midpoint = ['--midpoint-centered','--transition-scale-sd',str(width)]
quantiles = ['--crossover-quantile-low','0.10','--crossover-quantile-high','0.90']

def job_dir(name,stage,files,flags):
    ids = [file_hash(p) for p in files]
    spec = dict(revision=revision, inputs=ids, flags=flags)
    tag = hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()[:12]
    path = OUTPUT/name/stage/tag
    path.mkdir(exist_ok=True,parents=True)
    return path

def olambda_crossover():
    paths = [Path(OLAMBDA_TRANSITION_CSV)] if OLAMBDA_TRANSITION_CSV else sorted(O_ROOT.glob('*/model_result.csv'))
    valid = []
    for path in paths:
        root = path.parent
        if not (root/'notebook_config.json').is_file() or not (root/'run_manifest.json').is_file():
            continue
        config = json.loads((root/'notebook_config.json').read_text())
        manifest = json.loads((root/'run_manifest.json').read_text())
        prior = manifest.get('crossover_prior',{})
        if (config['input_sha256'] == {p.name:file_hash(p) for p in (oe,op,outcomes)}
            and manifest.get('hazard_activation_offset')==0.5 and manifest.get('adjustments')==adjustments
            and manifest.get('transition_scale_sd')==width
            and (prior.get('quantile_low'),prior.get('quantile_high'))==(0.10,0.90)):
            result = pd.read_csv(path)
            if len(result)==1 and result.crossover_median.iloc[0] > 1 and result.patients.iloc[0]==163:
                valid.append(path)
    if len(valid)!=1:
        raise ValueError(f'Expected one matching lifetime midpoint Oλ run; found {valid}. Set OLAMBDA_TRANSITION_CSV or finish the Oλ survival notebook.')
    return valid[0]

In [ ]:
keys = ['Dataset','sample_id']
est = pd.read_csv(oe).set_index(keys,verify_integrity=True)
assert len(est)==163
assert est.model.eq('latent_timescale_student_t3_scale_0.35_no_asymptote_floor').all()
assert (est.O_lambda_inf_central.gt(1) & np.isfinite(est.O_lambda_inf_central)).all()
p = pd.read_csv(op).set_index(keys,verify_integrity=True)
assert p.index.sort_values().equals(est.index.sort_values()) and p.shape[1]==16
assert (p.gt(1) & np.isfinite(p)).all().all()

features, individual_features = {}, {}
scores = ['TCAT_SCORE_ASA_CLR','TCAT_SCORE_Proliferation_CLR']
for name in RUN:
    csv,npz = inputs[name]
    with np.load(npz,allow_pickle=False) as archive:
        names = archive['feature_names'].astype(str).tolist()
    prefix = 'TCAT_GEP_' if name=='gep' else 'ATLAS_proportion_'
    features[name] = [f for f in names if f.startswith(prefix)]
    assert len(set(features[name]))==len(features[name])==(52 if name=='gep' else 11)
    individual_features[name] = features[name] + (scores if name=='gep' else [])
    assert set(individual_features[name]) <= set(names)
    audit = """
import sys, pandas as pd
from survival.survival_model import load_tcat_analysis
d, _, _, weights, _ = load_tcat_analysis(*sys.argv[1:4], 1000, ['Treatment','Disease'])
keys = ['Dataset','sample_id']
assert set(map(tuple,d[keys].values)) == set(map(tuple,pd.read_csv(sys.argv[4])[keys].values)), 'Patient identities differ'
assert weights.shape==(163,16)
"""
    subprocess.run([sys.executable,'-c',audit,csv,npz,outcomes,oe],cwd=repo,check=True)
    print(name, len(individual_features[name]), 'individual transitions;', len(features[name])+1, 'joint predictors')

In [ ]:
def joint(name, stage, o_cut=None):
    csv,npz = inputs[name]
    kind = 'gep' if name=='gep' else 'cell_type'
    mode = 'joint-gep' if name=='gep' else 'joint-cell-type'
    if stage=='continuous':
        module = 'survival.fit_tcat_survival'
        files = [csv,npz,outcomes,oe,op]
        flags = ['--mode',mode,'--olambda-estimates-csv',str(oe),'--olambda-particles-csv',str(op),*common]
        path = job_dir(name,'joint_continuous',files,flags)
        result_dir = path/mode
        args = [csv,npz,outcomes,path,*flags]
    else:
        module = 'survival.fit_tcat_joint_survival_regimes'
        files = [csv,npz,outcomes,oe,op,cuts[name],o_cut]
        flags = ['--feature-kind',kind,*midpoint,*common]
        path = job_dir(name,'joint_midpoint',files,flags)
        result_dir = path
        args = [*files,path,*flags]
    if not (result_dir/'joint_manifest.json').is_file():
        run_live([sys.executable,'-u','-m',module,*args,'--overwrite'])
    frame = pd.read_csv(result_dir/'joint_feature_results.csv').set_index('feature',verify_integrity=True)
    assert set(frame.index)==set(features[name])|{'O_lambda_inf'}
    assert frame.patients.eq(163).all() and frame.datasets.eq(8).all()
    assert (result_dir/'posterior.nc').stat().st_size
    if stage=='midpoint':
        manifest = json.loads((result_dir/'joint_manifest.json').read_text())
        assert manifest['hazard_activation_offset']==.5
        fixed = pd.read_csv(cuts[name]).set_index('feature').crossover_median.copy()
        fixed.loc['O_lambda_inf'] = pd.read_csv(o_cut).iloc[0].crossover_median
        np.testing.assert_allclose(frame.crossover_median_fixed,fixed.loc[frame.index],rtol=1e-12)
    print(f'{name} | joint {stage} | {result_dir}')
    display(frame[['HR_median','HR_q025','HR_q975','divergences','max_rhat','min_ess_bulk']])
    return result_dir

## Joint continuous fits
These use the new lifetime asymptotes and particles.

In [ ]:
continuous_dirs = {name:joint(name,'continuous') for name in RUN}

## Read existing feature-only midpoint crossovers
Select the complete table for each representation. Set FEATURE_CROSSOVERS if more than one table matches.

In [ ]:
cuts = {}
for name in RUN:
    paths = ([Path(FEATURE_CROSSOVERS[name])] if name in FEATURE_CROSSOVERS else
             sorted((FEATURE_ROOT / name / 'individual_midpoint').glob('*/regime_results.csv')))
    valid = []
    for path in paths:
        table = pd.read_csv(path).set_index('feature', verify_integrity=True)
        if set(features[name]).issubset(table.index):
            selected = table.loc[features[name]]
            if (np.isfinite(selected.crossover_median).all() and selected.hazard_activation_offset.eq(.5).all()
                and selected.patients.eq(163).all() and selected.datasets.eq(8).all()):
                valid.append(path)
    if len(valid) != 1:
        raise ValueError(f'{name}: found {len(valid)} complete midpoint tables. Set FEATURE_CROSSOVERS to the adjusted individual run: {valid}')
    cuts[name] = valid[0]
    print(name, cuts[name])


## Joint midpoint fits
Use the existing individual feature crossovers and the new lifetime Oλ crossover.

In [ ]:
o_cut = olambda_crossover()
print('Lifetime Oλ crossover:',o_cut)
transition_dirs = {name:joint(name,'midpoint',o_cut) for name in RUN}